<a href="https://colab.research.google.com/github/Chirag3841/Carbon_Emission_Prediction-Week2-/blob/main/Tree_Sitting_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
!pip install tree-sitter # install tree-sitter

In [62]:
!pip install tree-sitter-languages # install tree-sitter-languages
from tree_sitter import Parser # import parser
from tree_sitter_languages import get_language # import get language


In [63]:
PY_LANGUAGE = get_language("python")
parser = Parser()
parser.set_language(PY_LANGUAGE) # set language

In [64]:
def get_code_text(code, node): # get code text
    return code[node.start_byte:node.end_byte].decode("utf-8")

In [65]:
def extract_chunks(src_code):# extract chunks in form of functions,calls,import and classes
    src_bytes = src_code.encode()
    tree = parser.parse(src_bytes)
    root = tree.root_node

    chunks = {
        "functions": [], # chunks
        "classes": [],
        "imports": [],
        "calls": [],
    }

    def traverse(node):
        t = node.type

        # Functions
        if t == "function_definition":
            name = node.child_by_field_name("name")
            chunks["functions"].append({
                "name": get_code_text(src_bytes, name),
                "start": node.start_point,
                "end": node.end_point,
                "code": get_code_text(src_bytes, node)
            })

        # Classes
        elif t == "class_definition":
            name = node.child_by_field_name("name")
            chunks["classes"].append({
                "name": get_code_text(src_bytes, name),
                "start": node.start_point,
                "end": node.end_point,
                "code": get_code_text(src_bytes, node)
            })

        # Imports
        elif t in ("import_statement", "import_from_statement"):
            chunks["imports"].append({
                "code": get_code_text(src_bytes, node),
                "start": node.start_point,
                "end": node.end_point
            })

        # Function calls
        elif t == "call":
            fn = node.child_by_field_name("function")
            chunks["calls"].append({
                "name": get_code_text(src_bytes, fn),
                "start": node.start_point,
                "end": node.end_point,
                "code": get_code_text(src_bytes, node)
            })

        for child in node.children:
            traverse(child)

    traverse(root)
    return chunks


In [68]:
code= """

import math

def calculate_area_circle(radius):

    return math.pi * (radius ** 2)

class Shape:

    def __init__(self, name):
        self.name = name

    def get_info(self):
        return f"This is a {self.name}."

class Circle(Shape):

    def __init__(self, radius):
        super().__init__("Circle")
        self.radius = radius

    def get_area(self):
        return calculate_area_circle(self.radius)

    def get_info(self):
        return f"This is a {self.name} with radius {self.radius}."

def process_shapes(shapes_list):

    for shape in shapes_list:
        print(shape.get_info())
        if isinstance(shape, Circle):
            print(f"Area: {shape.get_area():.2f}")
        print("-" * 20)


from my_module import Circle, process_shapes

def main():

    circle1 = Circle(5)
    circle2 = Circle(10.5)

    shapes = [circle1, circle2]

    process_shapes(shapes)


    print(f"Area of a circle with radius 7: {my_module.calculate_area_circle(7):.2f}")

if __name__ == "__main__":
    main()
"""


In [69]:
import pprint
pprint.pprint(extract_chunks(code)) # readability enchancement


{'calls': [{'code': 'super().__init__("Circle")',
            'end': (19, 34),
            'name': 'super().__init__',
            'start': (19, 8)},
           {'code': 'super()',
            'end': (19, 15),
            'name': 'super',
            'start': (19, 8)},
           {'code': 'calculate_area_circle(self.radius)',
            'end': (23, 49),
            'name': 'calculate_area_circle',
            'start': (23, 15)},
           {'code': 'print(shape.get_info())',
            'end': (31, 31),
            'name': 'print',
            'start': (31, 8)},
           {'code': 'shape.get_info()',
            'end': (31, 30),
            'name': 'shape.get_info',
            'start': (31, 14)},
           {'code': 'isinstance(shape, Circle)',
            'end': (32, 36),
            'name': 'isinstance',
            'start': (32, 11)},
           {'code': 'print(f"Area: {shape.get_area():.2f}")',
            'end': (33, 50),
            'name': 'print',
            'start': (33, 1